# Credit Risk Scoring Engine — Fairness, Governance & Data Limitations

## Overview & Governance Framework
Fairness and ethical AI auditing are mandatory prerequisites before deploying machine learning credit scoring models in retail lending. This notebook evaluates model fairness, subgroup disparity metrics, and dataset limitations.

### Key Governance Boundaries
1. **Dataset Origin & Transferability Limitations**: The dataset comprises 1,000 historical German credit applicants from the 1970s. Historical, geographical, and temporal biases mean model scores **cannot be directly deployed** in contemporary retail markets (such as Indian retail lending) without re-estimation.
2. **Production Features vs. Audit Variables**: Sensitive demographic attributes (e.g., gender, marital status, age) should be audited for algorithmic bias rather than blindly used as primary credit criteria.
3. **Sample Size Reliability**: Subgroups with N < 25 are flagged with `[Warning] Small Subgroup Sample Size`. Empirical disparity metrics in tiny samples must not be interpreted as population estimates.
4. **Regulatory Disclaimers**: All findings require formal legal and compliance review under applicable lending statutes (e.g., RBI Fair Practice Code, ECOA, GDPR).

In [2]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, precision_score, recall_score, accuracy_score

# Load raw data for demographic attributes and processed data for modeling
raw_df = pd.read_csv('../data/raw/german_credit.csv')
X_test = pd.read_csv('../data/processed/X_test.csv')
y_test = pd.read_csv('../data/processed/y_test.csv').values.ravel()

with open('../models/threshold_config.json', 'r') as f:
    threshold_config = json.load(f)
optimal_threshold = threshold_config['optimal_threshold']

# Get test set indices (since split was 80/20 stratified random_state=42)
from sklearn.model_selection import train_test_split
_, test_idx = train_test_split(raw_df.index, test_size=0.20, stratify=raw_df['credit_risk'], random_state=42)
test_raw = raw_df.loc[test_idx].copy()

# Load tuned model
tuned_models = joblib.load('../models/tuned_models.joblib')
model = tuned_models[threshold_config['selected_model']]

# Predict probabilities and binary decisions at optimal threshold (0.35)
test_raw['prob_default'] = model.predict_proba(X_test)[:, 1]
test_raw['pred_default'] = (test_raw['prob_default'] >= optimal_threshold).astype(int)
test_raw['actual_default'] = y_test

print(f"Loaded {len(test_raw)} test applicants with model predictions (Threshold = {optimal_threshold}).")


In [3]:
sex_names = {
    'A91': 'Male: Divorced/Separated',
    'A92': 'Female: Divorced/Separated/Married',
    'A93': 'Male: Single',
    'A94': 'Male: Married/Widowed',
    'A95': 'Female: Single'
}
test_raw['sex_group'] = test_raw['personal_status_sex'].map(sex_names)

def audit_group(df_sub, group_col):
    records = []
    for grp, group_df in df_sub.groupby(group_col, observed=False):
        n = len(group_df)
        if n == 0:
            continue
        actual_bad = group_df['actual_default'].sum()
        pred_bad = group_df['pred_default'].sum()
        approved = (group_df['pred_default'] == 0).sum()
        approval_rate = (approved / n) * 100
        
        tn, fp, fn, tp = confusion_matrix(group_df['actual_default'], group_df['pred_default'], labels=[0, 1]).ravel()
        fpr = (fp / (fp + tn) * 100) if (fp + tn) > 0 else 0.0
        fnr = (fn / (fn + tp) * 100) if (fn + tp) > 0 else 0.0
        rec = (tp / (tp + fn) * 100) if (tp + fn) > 0 else 0.0
        prec = (tp / (tp + fp) * 100) if (tp + fp) > 0 else 0.0
        
        sample_flag = "[Warning] Small Sample" if n < 25 else "Stable"
        
        records.append({
            'Group': grp,
            'Applicants': n,
            'Approval Rate (%)': round(approval_rate, 2),
            'Pred Bad Rate (%)': round((pred_bad/n)*100, 2),
            'FPR (%)': round(fpr, 2),
            'FNR (%)': round(fnr, 2),
            'Recall (%)': round(rec, 2),
            'Precision (%)': round(prec, 2),
            'Sample Flag': sample_flag
        })
    return pd.DataFrame(records)

sex_fairness_df = audit_group(test_raw, 'sex_group')
print("=== Fairness Audit: Personal Status & Sex ===")
print(sex_fairness_df.to_string(index=False))


In [4]:
age_bins = [18, 25, 35, 45, 55, 100]
age_labels = ['18-25', '26-35', '36-45', '46-55', '56+']
test_raw['age_group'] = pd.cut(test_raw['age_in_years'], bins=age_bins, labels=age_labels, right=True)

age_fairness_df = audit_group(test_raw, 'age_group')
print("=== Fairness Audit: Age Groups ===")
print(age_fairness_df.to_string(index=False))


In [5]:
worker_names = {'A201': 'Foreign Worker (Yes)', 'A202': 'Foreign Worker (No / Native)'}
test_raw['worker_group'] = test_raw['foreign_worker'].map(worker_names)

worker_fairness_df = audit_group(test_raw, 'worker_group')
print("=== Fairness Audit: Foreign Worker Status ===")
print(worker_fairness_df.to_string(index=False))


In [6]:
print("=== Model Governance & Regulatory Compliance Summary ===\n")
gov_summary = [
    ("1. Production Feature Policy", "Demographic attributes ('personal_status_sex', 'foreign_worker') must NOT be used as scoring inputs in production to comply with non-discrimination mandates."),
    ("2. Small Sample Size Protection", "Subgroups with N < 25 (e.g. Female Single A95, Male Divorced A91) demonstrate wide variance. Metric differences in these cells are statistically un-anchored."),
    ("3. Disparate Impact Monitoring", "Approval rates vary across demographic groups (e.g., younger 18-25 applicants face lower approval rates due to shorter credit history). Disparate impact ratios require ongoing auditing."),
    ("4. Legal Review Requirement", "Deployment in any sovereign jurisdiction requires formal legal review under local credit statutes (e.g., Fair Credit Reporting Act, Equal Credit Opportunity Act, RBI Fair Practice Code).")
]
for header, text in gov_summary:
    print(f"{header}:\n  {text}\n")
